In [ ]:
from imagematerials.rest_of.sankey_function import create_node_dict, index_mapper, convert_index_to_node_id, prepare_Sankey_lists
from imagematerials.rest_of.fossil_fuels import *                                                                                            
from imagematerials.read_mym import read_mym_df

from imagematerials.rest_of.const import path_scenario_data_fossil, DIM2_primary_dict

In [89]:
def read_in_data(scenario: str):
    # Total Primary Energy Supply (TPES) in PJ per region by energy carrier, [NRCT, PRIM + 4](t), [28,13](t), # unit: PJ
    primary_energy_supply = read_mym_df(path_input_data.joinpath(scenario, 'EnergyFlows/tpes_ext.out')).set_index(["time", "DIM_1"])
    # Primary to Secondary energy flows DIM_1:Primary, DIM_2:Secondary GJ/yr
    prim_per_sec = read_mym_df(path_input_data.joinpath(scenario, 'EnergyFlows/PrimPerSec.out')).set_index(["time", "DIM_1", "DIM_2"])
    # Final Energy in PJ/yr by region, sector, and energy carrier [NRCT, S, NECS9T](t), [28,8,10](t) # PJ
    final_energy = read_mym_df(path_input_data.joinpath(scenario, 'EnergyFlows/final_energy_rt.out')).set_index(["time", "DIM_1", "DIM_2"])

    # Deal with the index and columns of all
    # Total Primary Energy Supply (TPES) in PJ/yr
    # TODO: region 27 has some values? Why? it is not exactly the sum of sth...
    primary_energy_supply = primary_energy_supply.T  # transpose to make DIM_2 the index (DIM_1 is regions, make this columns)
    primary_energy_supply = primary_energy_supply.stack('time')  # add years as index
    primary_energy_supply = primary_energy_supply.swaplevel()  # make years the first index
    primary_energy_supply = primary_energy_supply.sort_index()  # sort dataframe based on the index
    primary_energy_supply.index.names = ['time','primary_energy']
    # drop 27 and 28 regions from columns
    primary_energy_supply = primary_energy_supply.drop(columns=[27, 28])  # drop regions 27 and 28 if they exist
    # convert DIM_2 index numbers to energy carriers names with dict
    # Reverse the dict: number -> name
    dim2_reverse = {v: k for k, v in DIM2_primary_dict.items()}

    # If DIM2 is in the index:
    primary_energy_supply.index = primary_energy_supply.index.set_levels(
        primary_energy_supply.index.levels[primary_energy_supply.index.names.index('primary_energy')].map(dim2_reverse),
        level='primary_energy'
    )


   
    return primary_energy_supply, prim_per_sec, final_energy


In [90]:
primary_energy_supply, prim_per_sec, final_energy = read_in_data("SSP2_CP")  # replace with your scenario

C:\Users\Arp00003\AppData\Local\Temp\ipykernel_3768\2613512188.py:13: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  primary_energy_supply = primary_energy_supply.stack('time')  # add years as index


In [91]:
primary_energy_supply

DIM_1                           1            2             3             4   \
time primary_energy                                                           
1971 coal                 698.0271  12522.50000     62.545410      6.109577   
     oil                 3249.8190  30837.14000   1112.698000   1314.193000   
     nan                    0.0000      0.00000      0.000000      0.000000   
     natural gas         1089.0570  19398.21000    372.630400     67.445760   
     modern biofuel         0.0000      0.00000      0.000000      0.000000   
...                            ...          ...           ...           ...   
2100 solar PV            1317.0720  10518.99000   3482.165000   1165.722000   
     wind                2027.2930   9506.38800   1286.240000    649.070900   
     other renewables     113.8944    112.90520    215.670000    269.212000   
     hydro electricity    639.7349     13.43455      5.291054     29.729080   
     total              12037.4500  68972.36000  16070.150000  12709.050000   

DIM_1                            5           6            7              8   \
time primary_energy                                                           
1971 coal                 106.85150    192.3425     40.05804      12.624920   
     oil                 1141.50700   2572.9760    589.32530     372.332000   
     nan                    0.00000      0.0000      0.00000       0.000000   
     natural gas            6.42315    648.7023    184.52220       4.185012   
     modern biofuel         0.00000      0.0000      0.00000       0.000000   
...                             ...         ...          ...            ...   
2100 solar PV            5884.58200   3280.9690   8570.71800   22242.700000   
     wind                4416.85900   3906.3540   3317.08400    7959.845000   
     other renewables       0.00048    340.3232     39.79800       0.005738   
     hydro electricity   2534.36300   2370.5930     23.08436    1453.621000   
     total              32797.46000  27624.1900  38563.09000  106826.000000   

DIM_1                             9            10  ...            17  \
time primary_energy                                ...                 
1971 coal                   2.423960  1654.349000  ...      9.633130   
     oil                  224.370900   522.470800  ...   2467.381000   
     nan                    0.000000     0.000000  ...      0.000000   
     natural gas            0.004345     0.859017  ...    620.730300   
     modern biofuel         0.000000     0.000000  ...      0.000000   
...                              ...          ...  ...           ...   
2100 solar PV           11166.020000   996.000900  ...  13612.310000   
     wind                5173.445000  1267.983000  ...   3543.461000   
     other renewables    1972.157000   256.108400  ...      0.405584   
     hydro electricity    319.464000    30.102670  ...     15.411200   
     total              46170.370000  8635.906000  ...  59048.910000   

DIM_1                             18           19             20           21  \
time primary_energy                                                             
1971 coal                 1303.67500   944.474500    6312.123000    212.05430   
     oil                   860.34010   482.178800    2319.794000   1414.14300   
     nan                     0.00000     0.000000       0.000000      0.00000   
     natural gas            32.37033     1.719365     159.578600     11.59313   
     modern biofuel          0.00000     0.000000       0.000000      0.00000   
...                              ...          ...            ...          ...   
2100 solar PV            16558.21000   323.211700   12149.670000   5358.75300   
     wind                 4819.39600   213.265000   14114.640000   2078.85000   
     other renewables       76.70958    11.141060       1.717159    353.93620   
     hydro electricity     120.18070     3.191360    4095.674000    111.66350   
     total              134975.2000

In [15]:
# -*- coding: utf-8 -*-
"""
Created on Tue May  7 13:15:30 2024

@author: Arp00003
"""

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import plotly.graph_objects as go

from imagematerials.read_mym import read_mym_df
from imagematerials.rest_of.const import (parse_dim, get_key, DIM2_primary_dict, 
                                          path_figures, path_input_data)

from imagematerials.rest_of.sankey_function import create_node_dict, index_mapper, convert_index_to_node_id, prepare_Sankey_lists

coal_conversion = 28.9 # coal: 28.9 MJ/kg Coal (low bituminous)
oil_conversion = 45.5 # oil: 45.5 MJ/kg Crude oil
gas_conversion = 52.2 # gas: 52.2 MJ/kg Natural gas (95% methane)

mega_to_peta = 1e9
giga_to_peta = 1e6

#%% Primary energy from Joule to kg
def converte_primary_energy_to_mass(fossils_primary):
    coal_converted = fossils_primary.query(parse_dim('primary', '2', 'coal')) * mega_to_peta / coal_conversion
    oil_converted = fossils_primary.query(parse_dim('primary', '2', 'oil')) * mega_to_peta / oil_conversion
    gas_converted = fossils_primary.query(parse_dim('primary', '2', 'natural gas')) * mega_to_peta / gas_conversion
    
    fossils_primary_converteted = pd.concat([coal_converted, oil_converted, gas_converted])
    
    return fossils_primary_converteted




def convert_primary_to_secondary_to_mass(fossils_prim_per_sec):
    coal_converted = fossils_prim_per_sec.query(parse_dim('primsec_reversed', '1', 'coal')) * mega_to_peta / coal_conversion
    conv_oil_converted = fossils_prim_per_sec.query(parse_dim('primsec_reversed', '1', 'conventional oil')) * mega_to_peta / oil_conversion
    unconv_oil_converted = fossils_prim_per_sec.query(parse_dim('primsec_reversed', '1', 'unconventional oil')) * mega_to_peta / oil_conversion
    gas_converted = fossils_prim_per_sec.query(parse_dim('primsec_reversed', '1', 'natural gasses')) * mega_to_peta / gas_conversion
    
    fossils_primsecond_converted = pd.concat([coal_converted, conv_oil_converted, unconv_oil_converted, gas_converted])
    
    return fossils_primsecond_converted
    


def convert_secondary_to_final_mass(fossils_final):
    coal_converted = fossils_final.query(parse_dim('seconden_reversed', '3', 'coal')) * mega_to_peta / coal_conversion
    coal_converted.drop(columns=[27, 28], inplace=True)  # drop 27 and 28, which are empty regions & global region

    heavy_oil_converted = fossils_final.query(parse_dim('seconden_reversed', '3', 'heavy oil')) * mega_to_peta / oil_conversion
    heavy_oil_converted.drop(columns=[27, 28], inplace=True)  

    light_oil_converted = fossils_final.query(parse_dim('seconden_reversed', '3', 'light oil')) * mega_to_peta / oil_conversion
    light_oil_converted.drop(columns=[27, 28], inplace=True)

    gas_converted = fossils_final.query(parse_dim('seconden_reversed', '3', 'natural gas')) * mega_to_peta / gas_conversion
    gas_converted.drop(columns=[27, 28], inplace=True)

    return {'coal': coal_converted, 
            'heavy oil': heavy_oil_converted, 
            'light oil': light_oil_converted, 
            'natural gas': gas_converted }



def fossil_fuel_data(scenario):
    # https://www.engineeringtoolbox.com/fossil-fuels-energy-content-d_1298.html
    # read in and format relevant IMAGE data

    # Total Primary Energy Supply (TPES) in PJ per region by energy carrier, [NRCT, PRIM + 4](t), [28,13](t), # unit: PJ
    primary_energy_supply = read_mym_df(path_input_data.joinpath(scenario, 'EnergyFlows/tpes_ext.out')).set_index(["time", "DIM_1"])
    # Primary to Secondary energy flows DIM_1:Primary, DIM_2:Secondary GJ/yr
    prim_per_sec = read_mym_df(path_input_data.joinpath(scenario, 'EnergyFlows/PrimPerSec.out')).set_index(["time", "DIM_1", "DIM_2"])
    # Final Energy in PJ/yr by region, sector, and energy carrier [NRCT, S, NECS9T](t), [28,8,10](t) # PJ
    final_energy = read_mym_df(path_input_data.joinpath(scenario, 'EnergyFlows/final_energy_rt.out')).set_index(["time", "DIM_1", "DIM_2"])  

    # Total Primary Energy Supply (TPES) in PJ/yr
    # TODO: region 27 has some values? Why? it is not exactly the sum of sth...
    primary_energy_supply = primary_energy_supply.T  # transpose to make DIM_2 the index (DIM_1 is regions, make this columns)
    primary_energy_supply = primary_energy_supply.stack('time')  # add years as index
    primary_energy_supply = primary_energy_supply.swaplevel()  # make years the first index
    primary_energy_supply = primary_energy_supply.sort_index()  # sort dataframe based on the index
    primary_energy_supply.index.names = ['time','DIM_2']

    # Primary to Secondary energy flows in GJ/yr
    prim_per_sec = prim_per_sec/giga_to_peta # GJ to PJ
    prim_per_sec = prim_per_sec.rename(columns={27: 28}) # rename 27 to 28 to fit other dfs
    prim_per_sec[27] = np.nan # create empty dummy 27 region
    prim_per_sec = prim_per_sec[sorted(prim_per_sec.columns)] # sort so that 27 is before 28

    # Final Energy in PJ/yr 
    #TODO: region 27 is empty
    final_energy = final_energy.T # transpose to make DIM_1 the index
    final_energy = final_energy.stack(['time', 'DIM_2'])  # add years as and DIM_2 index
    final_energy.index.names = ['DIM_3', 'time', 'DIM_2']
    final_energy = final_energy.swaplevel(0, 1)  # make years the first index

    #%% Fossil energy only

    fossils_primary = primary_energy_supply.query(parse_dim('primary', '2', 'coal' ,'oil', 
                                                    'natural gas'))
    fossils_prim_per_sec = prim_per_sec.query(parse_dim('primsec_reversed', '1', 'coal', 
                                                'conventional oil', 'unconventional oil', 
                                                'natural gasses'))
    fossils_final = final_energy.query(parse_dim('seconden_reversed', '3', 'coal',
                                            'heavy oil', 'light oil', 'natural gas'))
    
    fossils_primary_converted = converte_primary_energy_to_mass(fossils_primary)
    fossils_primsecond_converted = convert_primary_to_secondary_to_mass(fossils_prim_per_sec)
    fossils_final_converteted = convert_secondary_to_final_mass(fossils_final)

    return_dict = {
        'fossils_primary': fossils_primary,
        'fossils_prim_per_sec': fossils_prim_per_sec,
        'fossils_final': fossils_final,
        'fossils_primary_converted': fossils_primary_converted,
        'fossils_primsecond_converted': fossils_primsecond_converted,
        'fossils_final_converteted': fossils_final_converteted
    }

    return return_dict



In [16]:
#%% Plot primary energy of selected region (in joule or kg)

def plot_stacked_fossils(country_id: int, fossils_primary: pd.DataFrame, unit: str):
    fossils_primary_region = fossils_primary.loc[:, country_id].unstack() # unstacks second index (primary energy types)
    
    fig, ax = plt.subplots(figsize=(15, 10))
    # fossils_primary_region.plot(kind='area', ax=ax)
    
    # define labels via dict
    labels = [get_key(i, DIM2_primary_dict) for i in list(fossils_primary_region.columns)]
    
    # Add labels and title
    ax.stackplot(fossils_primary_region.index, 
                 fossils_primary_region.loc[:, 1], 
                 fossils_primary_region.loc[:, 2], 
                 fossils_primary_region.loc[:, 4],
                 labels = labels)
    
    # .rename(get_key(1, DIM2_primary_dict))
    ax.legend(loc='upper left')
    ax.set_xlabel('Years')
    ax.set_ylabel(f'Fossil energy [{unit}]')
    
    plt.xlim(left=1971, right=2100)
    fig.tight_layout()

#%% plot primary per secondary sankey 

In [17]:
def plot_fossils_sankey(year: int, country_id: int, df1: pd.DataFrame, 
                        df2: pd.DataFrame, DIM1_primsec_dict, DIM2_primsec_dict, 
                        DIM3_seconden_dict, DIM2_sectors_dict, unit: str):

    # Create nodes
    energy_sources = [*DIM1_primsec_dict.values(), *DIM2_primsec_dict.values(),
                      *DIM3_seconden_dict.values(), *DIM2_sectors_dict.values()
                      ]  # * flattens the dict_keys objects
    nodes = create_node_dict(energy_sources)
    
    # Prepare dataframes
    # Get "real" name of energy sources instead of their numbers
    fossils_prim_per_sec_sankey = df1.loc[year, country_id].copy()
    fossils_prim_per_sec_sankey.index = fossils_prim_per_sec_sankey.index.map(index_mapper(
        DIM1_primsec_dict.get,
        DIM2_primsec_dict.get
    ))
    
    fossils_final_energy_sankey = df2.loc[year, country_id].copy()
    fossils_final_energy_sankey.index = fossils_final_energy_sankey.index.map(index_mapper(
        DIM3_seconden_dict.get,
        DIM2_sectors_dict.get
    ))
    
    # Drop where either dimension is "total"
    is_total = fossils_prim_per_sec_sankey.index.get_level_values(0).isin(["total"]) | fossils_prim_per_sec_sankey.index.get_level_values(1).isin(["total"])
    fossils_prim_per_sec_sankey = fossils_prim_per_sec_sankey[~is_total]
    
    is_total = fossils_final_energy_sankey.index.get_level_values(0).isin(["total"]) | fossils_final_energy_sankey.index.get_level_values(1).isin(["total"])
    fossils_final_energy_sankey = fossils_final_energy_sankey[~is_total]
    
    # Convert "real" names to unique node identifiers
    convert_index_to_node_id(fossils_prim_per_sec_sankey, nodes)
    convert_index_to_node_id(fossils_final_energy_sankey, nodes)
    
    # Prepare lists for Sankey diagram
    link_source, link_target, link_value = prepare_Sankey_lists(fossils_prim_per_sec_sankey)
    prepare_Sankey_lists(fossils_final_energy_sankey, link_source, link_target, link_value)
    
    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey( 
        node = dict( 
          thickness = 5, 
          line = dict(color = "green", width = 0.1), 
          label = list(nodes.keys()), 
          color = "blue"
        ), 
        link = dict(
            source = link_source,  
            target = link_target, 
            value = link_value
      ))]) 
    
    # fig.write_image("figures/energy_sankey.jpg")
    fig.write_html(f"{path_figures}/fossils_global_{unit}.html")

In [18]:
# # https://www.engineeringtoolbox.com/fossil-fuels-energy-content-d_1298.html



# #%% read in and format relevant IMAGE data

# # Total Primary Energy Supply (TPES) in PJ per region by energy carrier, [NRCT, PRIM + 4](t), [28,13](t), # unit: PJ
# primary_energy_supply = read_mym_df(f'{path_scenario_data_fossil}/tpes_ext.out').set_index(["time", "DIM_1"])
# # Primary to Secondary energy flows DIM_1:Primary, DIM_2:Secondary GJ/yr
# prim_per_sec = read_mym_df(f'{path_scenario_data_fossil}/PrimPerSec.out').set_index(["time", "DIM_1", "DIM_2"])
# # Final Energy in PJ/yr by region, sector, and energy carrier [NRCT, S, NECS9T](t), [28,8,10](t) # PJ
# final_energy = read_mym_df(f'{path_scenario_data_fossil}/final_energy_rt.out').set_index(["time", "DIM_1", "DIM_2"])  

# # Total Primary Energy Supply (TPES) in PJ/yr
# # TODO: region 27 has some values? Why? it is not exactly the sum of sth...
# primary_energy_supply = primary_energy_supply.T  # transpose to make DIM_2 the index (DIM_1 is regions, make this columns)
# primary_energy_supply = primary_energy_supply.stack('time')  # add years as index
# primary_energy_supply = primary_energy_supply.swaplevel()  # make years the first index
# primary_energy_supply = primary_energy_supply.sort_index()  # sort dataframe based on the index
# primary_energy_supply.index.names = ['time','DIM_2']

# # Primary to Secondary energy flows in GJ/yr
# prim_per_sec = prim_per_sec/giga_to_peta # GJ to PJ
# prim_per_sec = prim_per_sec.rename(columns={27: 28}) # rename 27 to 28 to fit other dfs
# prim_per_sec[27] = np.nan # create empty dummy 27 region
# prim_per_sec = prim_per_sec[sorted(prim_per_sec.columns)] # sort so that 27 is before 28

# # Final Energy in PJ/yr 
# #TODO: region 27 is empty
# final_energy = final_energy.T # transpose to make DIM_1 the index
# final_energy = final_energy.stack(['time', 'DIM_2'])  # add years as and DIM_2 index
# final_energy.index.names = ['DIM_3', 'time', 'DIM_2']
# final_energy = final_energy.swaplevel(0, 1)  # make years the first index

# #%% Fossil energy only

# fossils_primary = primary_energy_supply.query(parse_dim('primary', '2', 'coal' ,'oil', 
#                                                         'natural gas'))
# fossils_prim_per_sec = prim_per_sec.query(parse_dim('primsec_reversed', '1', 'coal', 
#                                                     'conventional oil', 'unconventional oil', 
#                                                     'natural gasses'))
# fossils_final = final_energy.query(parse_dim('seconden_reversed', '3', 'coal',
#                                              'heavy oil', 'light oil', 'natural gas'))

In [19]:
fossil_fuel_data = fossil_fuel_data("SSP2_CP")  # replace with your scenario


C:\Users\Arp00003\AppData\Local\Temp\ipykernel_3768\1937868047.py:86: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  primary_energy_supply = primary_energy_supply.stack('time')  # add years as index
C:\Users\Arp00003\AppData\Local\Temp\ipykernel_3768\1937868047.py:100: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  final_energy = final_energy.stack(['time', 'DIM_2'])  # add years as and DIM_2 index


In [20]:
fossil_fuel_data

{'fossils_primary': DIM_1              1           2           3            4            5   \
 time DIM_2                                                                
 1971 1       698.0271  12522.5000    62.54541     6.109577    106.85150   
      2      3249.8190  30837.1400  1112.69800  1314.193000   1141.50700   
      4      1089.0570  19398.2100   372.63040    67.445760      6.42315   
 1972 1       617.3547  13549.5400    69.98534     6.253846    113.25750   
      2      3415.9660  32561.9600  1222.90100  1378.695000   1262.60300   
 ...               ...         ...         ...          ...          ...   
 2099 2      1855.2090  12523.6100  3219.88300  1786.765000   4248.85000   
      4      5651.6480  31496.9000  5692.82500  3023.600000  13905.74000   
 2100 1       117.7780    251.0827   834.22810  3783.747000    226.53780   
      2      1850.7760  12439.3400  3155.66300  1777.086000   4176.35600   
      4      5557.0410  30904.0900  5611.31600  3063.398000  13776.55

In [37]:
final_energy = read_mym_df(path_input_data.joinpath("SSP2_CP", 'EnergyFlows/final_energy_rt.out')).set_index(["time", "DIM_1", "DIM_2"])  
# final_energy = final_energy.T # transpose to make DIM_1 the index
# final_energy = final_energy.stack(['time', 'DIM_2'])  # add years as and DIM_2 index
# final_energy.index.names = ['DIM_3', 'time', 'DIM_2']
# final_energy = final_energy.swaplevel(0, 1)  # make years the first index

final_energy

1             2             3            4  \
time DIM_1 DIM_2                                                          
1971 1     1      1.885407e+02    378.652400      23.19438     465.6127   
           2      1.949883e+00    150.974500    1035.98600       0.0000   
           3      2.400606e+01    476.148100      75.83350     162.2789   
           4      9.603902e-24    337.471200      11.36705     212.4781   
           5      0.000000e+00     -0.316783      57.21869       0.0000   
...                        ...           ...           ...          ...   
2100 28    5      6.118241e+02    523.645000     339.24610     687.2070   
           6      1.765552e+03      0.000000   25457.60000   26204.2400   
           7      0.000000e+00      0.000000   60936.83000       0.0000   
           8      0.000000e+00    252.625300     306.07770       0.0000   
           9      1.758310e+04  31367.180000  120877.30000  117103.0000   

                             5             6             7             8  \
time DIM_1 DIM_2                                                           
1971 1     1      2.412652e+02  0.000000e+00  0.000000e+00  0.000000e+00   
           2      1.477286e-10  0.000000e+00  1.943009e-09  0.000000e+00   
           3      0.000000e+00  6.000206e+01  0.000000e+00  0.000000e+00   
           4      0.000000e+00  3.119263e-23  9.407405e-24  6.012254e-24   
           5      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
...                        ...           ...           ...           ...   
2100 28    5      2.955369e+03  6.186953e+01  9.432570e+00  2.575748e+01   
           6      1.071213e+04  0.000000e+00  0.000000e+00  0.000000e+00   
           7      2.999594e+03  0.000000e+00  1.853394e-04  0.000000e+00   
           8      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
           9      5.050869e+04  1.355491e+04  1.596859e+03  2.995943e+03   

                              9           10  
time DIM_1 DIM_2                              
1971 1     1         347.450400    1644.7160  
           2           7.740280    1196.6510  
           3         129.779600     928.0482  
           4         161.054800     722.3712  
           5           6.566401      63.4683  
...                         ...          ...  
2100 28    5        8910.502000   14124.8500  
           6           0.000000   64139.5200  
           7           0.391718   63936.8200  
           8         545.680700    1104.3840  
           9      397971.500000  753558.6000  

[32760 rows x 10 columns]

In [30]:
fossil_fuel_data.get("fossils_final_converteted").get('coal').loc[1971]

DIM_1                  1             2             3             4   \
DIM_3 DIM_2                                                           
1     1      6.523900e+09  9.236889e+10  1.661405e+09  2.079410e+08   
      2      6.747000e+07  3.958775e+05  1.444133e+04  2.665093e+03   
      3      8.306595e+08  1.576198e+10  0.000000e+00  0.000000e+00   
      4      3.323149e-16  8.259727e+09  0.000000e+00  1.025290e-10   
      5      0.000000e+00  1.171164e+10  0.000000e+00  6.184003e+06   
      6      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
      7      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
      8      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
      9      7.422031e+09  1.281026e+11  1.661420e+09  2.141276e+08   

DIM_1                  5             6             7             8   \
DIM_3 DIM_2                                                           
1     1      9.404294e+08  2.735446e+09  6.976491e+08  3.473616e+08   
      2      1.497524e+07  2.505088e+08  6.653021e+04  1.182855e+08   
      3      0.000000e+00  9.300401e+08  2.800123e+07  2.205911e+06   
      4      9.212689e-37  2.581857e+07  4.176869e+07  5.794969e+03   
      5      0.000000e+00  9.573692e+07  7.785433e+05  0.000000e+00   
      6      6.936439e+07  6.574391e+07  0.000000e+00  0.000000e+00   
      7      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
      8      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
      9      1.024769e+09  4.103294e+09  7.682640e+08  4.678588e+08   

DIM_1                  9             10  ...            17            18  \
DIM_3 DIM_2                              ...                               
1     1      1.875064e+07  1.233781e+10  ...  1.441893e+08  1.445195e+10   
      2      6.495066e+07  4.664803e+09  ...  4.053048e+03  8.963436e+09   
      3      0.000000e+00  2.523251e+09  ...  0.000000e+00  2.448484e+09   
      4      2.972078e+03  1.208548e+09  ...  0.000000e+00  3.646433e+09   
      5      0.000000e+00  1.111599e+08  ...  0.000000e+00  1.548697e+09   
      6      0.000000e+00  1.083576e+09  ...  0.000000e+00  0.000000e+00   
      7      0.000000e+00  0.000000e+00  ...  0.000000e+00  0.000000e+00   
      8      0.000000e+00  0.000000e+00  ...  0.000000e+00  0.000000e+00   
      9      8.370426e+07  2.192915e+10  ...  1.441934e+08  3.105900e+10   

DIM_1                  19            20            21            22  \
DIM_3 DIM_2                                                           
1     1      1.679070e+10  1.201162e+11  7.414561e+08  2.049131e+08   
      2      4.161792e+07  9.188654e+09  0.000000e+00  4.445730e+07   
      3      5.248436e+09  5.129318e+10  1.424607e+08  0.000000e+00   
      4      1.327837e+08  1.710256e+09  1.338254e+03  0.000000e+00   
      5      4.262557e+09  1.286483e+10  1.393308e+09  2.123487e+06   
      6      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
      7      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
      8      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
      9      2.647609e+10  1.951731e+11  2.277226e+09  2.514939e+08   

DIM_1                  23            24            25            26  
DIM_3 DIM_2                                                          
1     1      2.441080e+10  8.315228e+09  1.593309e+09  2.014799e+09  
      2      8.468522e+08  1.191347e+08  2.976118e+05  3.996062e+08  
      3      5.905353e+09  5.615045e+08  0.000000e+00  3.157532e+07  
      4      1.421475e+09  1.044453e+08  5.130308e-30  1.671728e+08  
      5      3.382007e+07  5.380398e+07  1.372562e+08  1.466564e+08  
      6      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  
      7      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  
      8      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  
      9      3.261830e+10  9.154118e+09  1.730863e+09  2.759809e+09  

[9 rows x 26 columns]